# Bài 20/25 · LeetCode #110 — Balanced Binary Tree

**Mức độ:** Easy · **Chủ đề:** Tree · Bottom-up DFS · Sentinel · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/balanced-binary-tree/](https://leetcode.com/problems/balanced-binary-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cây cân bằng chiều cao nếu **mọi node** có độ cao hai nhánh trái/phải khác nhau nhiều nhất 1. Trả boolean. Không đủ nếu chỉ kiểm tra gốc.

### Ví dụ quan trọng

`[3,9,20,None,None,15,7] → True`; `[1,2,2,3,3,None,None,4,4] → False`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [1]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. `TreeNode` thật sự là gì? Học từ một nút đơn

**Binary Tree (cây nhị phân)** gồm các nút (*node*). Mỗi nút lưu đúng ba thông tin:
`val` (giá trị), `left` (tham chiếu đến con trái), `right` (tham chiếu đến con phải).

Quan trọng: `left`/`right` **không phải** là vị trí trong mảng; đó là liên kết đến một **đối tượng** `TreeNode` khác.
`None` nghĩa là *không có nút*. Một cây rỗng được biểu diễn bởi `root = None`.

Ví dụ cây `1` có con trái `2` và con phải `3`:

```text
      1
     / \
    2   3
```

Nhìn vào code dựng cây dưới đây, hãy dự đoán từng phép in trước khi chạy.

In [2]:
root_demo = TreeNode(1)
root_demo.left = TreeNode(2)
root_demo.right = TreeNode(3)
print("Gốc:", root_demo.val)
print("Con trái:", root_demo.left.val)
print("Con phải:", root_demo.right.val)
print("Con trái của nút 2:", root_demo.left.left)  # None

Gốc: 1
Con trái: 2
Con phải: 3
Con trái của nút 2: None


### 2.2. Tham chiếu (*reference*) và giá trị khác nhau thế nào?

`a = root_demo.left` **không sao chép** một nút mới: `a` và `root_demo.left` là hai tên cùng chỉ đến một đối tượng.
Vì vậy `a.val = 99` sẽ thay đổi giá trị trong cây gốc.

- `a is b`: hỏi **có cùng một đối tượng** hay không.
- `a.val == b.val`: hỏi **hai giá trị bằng nhau** hay không.
- Hai nút khác nhau có thể cùng giá trị; với so sánh cây, ta cần xét cả cấu trúc, không chỉ xét một con số.

Phần này đặc biệt quan trọng khi gắn nút con trong giải pháp chính.

In [3]:
x_demo = TreeNode(7)
y_demo = TreeNode(7)
z_demo = x_demo
print("Giá trị bằng nhau:", x_demo.val == y_demo.val)  # True
print("Là một đối tượng:", x_demo is y_demo)        # False
print("Chỉ đến cùng nút:", x_demo is z_demo)       # True
z_demo.val = 8
print("Giá trị của x sau khi đổi qua z:", x_demo.val)

Giá trị bằng nhau: True
Là một đối tượng: False
Chỉ đến cùng nút: True
Giá trị của x sau khi đổi qua z: 8


### 2.3. `None`, `is None` và thứ tự kiểm tra an toàn

Với `node = None`, viết `node.val` sẽ gây `AttributeError`: không thể truy cập `val` của vật không tồn tại.
Cách an toàn: kiểm tra `node is None` **trước**, rồi mới dùng `node.val`.

Trong Python, biểu thức `A and B` có **short-circuit** (đoản mạch): nếu `A` là `False` thì Python **không đánh giá** `B` nữa.
Tương tự, với `A or B`, nếu `A` là `True` thì không cần tính `B`.
Đây là lý do ta có thể xử lý trường hợp rỗng trước khi truy cập thuộc tính và tránh nhánh đệ quy vô ích.

In [4]:
nothing = None
print("Là nút rỗng?", nothing is None)
print("Không phải nút rỗng?", nothing is not None)
print("Đọc an toàn:", nothing.val if nothing is not None else "không có nút")
leaf_demo = TreeNode(42)
print("Lá có hai con rỗng:", leaf_demo.left is None and leaf_demo.right is None)

Là nút rỗng? True
Không phải nút rỗng? False
Đọc an toàn: không có nút
Lá có hai con rỗng: True


### 2.4. Hiểu danh sách level-order và vì sao phải giữ `None`

LeetCode thường hiển thị cây theo thứ tự từng tầng, trái trước phải sau (*level-order*).
Trong notebook, hàm `tree(values)` dựng cây từ dạng ấy để dễ thử, còn `show_tree(root)` đưa cây về danh sách để đọc.
**Đừng coi list đó là định nghĩa của cây**: thuật toán nộp LeetCode nhận `TreeNode`, không nhận danh sách.

Ví dụ `[1,2,None,3]` là: gốc 1 có con trái 2, nút 2 có con trái 3.
Nó khác `[1,None,2,3]`: gốc 1 có con phải 2, nút 2 có con trái 3.
Dù phần tử `3` xuất hiện cuối cả hai danh sách, vị trí con trỏ khác nhau hoàn toàn.

In [5]:
shape_a = tree([1, 2, None, 3])
shape_b = tree([1, None, 2, 3])
print("A:", show_tree(shape_a), "gốc trái:", shape_a.left)
print("B:", show_tree(shape_b), "gốc trái:", shape_b.left)
print("A.gốc.phải:", shape_a.right, "B.gốc.phải:", shape_b.right)
assert show_tree(shape_a) != show_tree(shape_b)

A: [1, 2, None, 3] gốc trái: TreeNode(2)
B: [1, None, 2, 3] gốc trái: None
A.gốc.phải: None B.gốc.phải: TreeNode(2)


### 2.5. Đệ quy (*recursion*) từ con số 0: hàm gọi lại chính nó

Đệ quy có hai mảnh **bắt buộc**:

1. **Base case** (trường hợp dừng): đơn giản đến mức trả lời ngay, không gọi tiếp.
2. **Recursive step** (bước rút gọn): gọi lại hàm trên bài toán nhỏ hơn, rồi dùng kết quả ấy.

**Call stack (ngăn xếp lời gọi)** giữ lại vị trí mà mỗi lần gọi đang chờ `return`.
Ví dụ `countdown(3)` cần đi `3 → 2 → 1 → 0` rồi quay trở ra.
Trên cây, bước nhỏ hơn tự nhiên là cây con `node.left` hoặc `node.right`.

Kí hiệu chiều cao `h`: một đường từ gốc xuống lá dài tối đa `h` nút, nên tầng gọi lồng nhau có thể đến `h`.
Đệ quy không tự biến mất bộ nhớ: mỗi lần gọi đang chờ chiếm một phần call stack.

In [6]:
def countdown_demo(n, level=0):
    print("  " * level + f"vào n={n}")
    if n == 0:
        print("  " * level + "đã đến base case")
        return 0
    result = 1 + countdown_demo(n - 1, level + 1)
    print("  " * level + f"trở lại n={n}, trả {result}")
    return result

assert countdown_demo(3) == 3

vào n=3
  vào n=2
    vào n=1
      vào n=0
      đã đến base case
    trở lại n=1, trả 1
  trở lại n=2, trả 2
trở lại n=3, trả 3


### 2.6. `Optional[TreeNode]` và `return` là gì?

`Optional[TreeNode]` (trong `typing`) mô tả kiểu dữ liệu có thể là một `TreeNode` **hoặc** `None`.
Đây là **type hint** (gợi ý kiểu), không làm Python tự kiểm tra cây hợp lệ trong lúc chạy.

`return` *trả kết quả về nơi gọi và kết thúc lượt chạy hiện tại*. `print` chỉ đưa thông tin lên màn hình;
một hàm chỉ `print(True)` mà không `return True` sẽ trả về `None` (không đúng kiểu `bool` mà nhiều bài đòi).

Vòng lặp `for`, phép `max`, phép `abs`, danh sách `[]`, biểu thức `x is None`, phép truy cập `.left` / `.right`
đều sẽ được giải thích lại ngay tại chỗ cần dùng ở phần sau.

In [7]:
def check_node_demo(node: Optional[TreeNode]) -> bool:
    if node is None:
        return False
    return node.val > 0

print("Kiểm tra None:", check_node_demo(None))
print("Kiểm tra nút 5:", check_node_demo(TreeNode(5)))
print("Kiểm tra nút -2:", check_node_demo(TreeNode(-2)))

Kiểm tra None: False
Kiểm tra nút 5: True
Kiểm tra nút -2: False


## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Gọi hàm trả **hai loại thông tin**: chiều cao bình thường `0,1,...` hoặc `-1` đại diện trạng thái "không cân bằng" (sentinel). Kiểm tra `abs(a-b)` là độ chênh tuyệt đối.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [8]:
print('abs(2-3)=',abs(2-3),'abs(2-5)=',abs(2-5))
for x in [0,1,2,3]:print(x,'khác 1 bao nhiêu?',abs(x-1))

abs(2-3)= 1 abs(2-5)= 3
0 khác 1 bao nhiêu? 1
1 khác 1 bao nhiêu? 0
2 khác 1 bao nhiêu? 1
3 khác 1 bao nhiêu? 2


### 3.1. “Balanced” nghĩa là cân bằng **ở mọi nút**, không phải chỉ ở gốc

Một cây **height-balanced** nếu, với **từng nút bất kỳ**, độ cao cây con trái và cây con phải
chênh nhau **không quá 1**: `abs(h_left-h_right) <= 1`.

**Quy ước:** cây rỗng `None` có chiều cao 0; nút lá có chiều cao 1.

Chỉ nhìn hai nhánh của **gốc** là chưa đủ. Hãy xem cây hai phía cùng cao 3 nhưng bên trong có nút
bị lệch trái sâu 2 so với phải 0: gốc có thể trông cân, cây vẫn phải trả `False`.
Phần minh họa dưới cố ý tạo phản ví dụ mà *gốc cân nhưng cây con mất cân bằng*.

In [9]:
bad_but_root_equal = TreeNode(1,
    TreeNode(2, TreeNode(3, TreeNode(4))),
    TreeNode(5, TreeNode(6, TreeNode(7))))
print("Cây phản ví dụ:", show_tree(bad_but_root_equal))
print("Gốc có hai nhánh cao như nhau, nhưng cả hai nhánh lại lệch sâu.")

Cây phản ví dụ: [1, 2, 5, 3, None, 6, None, 4, None, 7]
Gốc có hai nhánh cao như nhau, nhưng cả hai nhánh lại lệch sâu.


### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Với mỗi node, gọi hàm `height` tính chiều cao của cả hai cây con, rồi lại làm tương tự ở con. Cách này dễ hiểu nhưng lặp đi lặp lại nhiều phép tính.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [10]:
def baseline(root):
    def height(n):return 1+max(height(n.left),height(n.right)) if n else 0
    if root is None:return True
    if abs(height(root.left)-height(root.right))>1:return False
    return baseline(root.left) and baseline(root.right)

In [11]:
for a in [[3,9,20,None,None,15,7],[1,2,2,3,3,None,None,4,4],[]]:
    print(a,'->',baseline(tree(a)))

[3, 9, 20, None, None, 15, 7] -> True
[1, 2, 2, 3, 3, None, None, 4, 4] -> False
[] -> True


### 4.1. Mổ xẻ baseline — đúng nhưng tính chiều cao lặp lại nhiều lần

`baseline` dùng hai hàm:
- `height(node)` tính độ cao của một cây con bằng `1+max(...)`.
- `baseline(node)` kiểm tra chênh chiều cao rồi gọi lại `baseline` cho hai cây con.

Khi đứng ở gốc, `height(root.left)` đã đi qua mọi nút ở nhánh trái.
Sau đó `baseline(root.left)` **lại** tính chiều cao các nhánh con, có nghĩa nhiều nút bị thăm đi thăm lại.

Với **cây cân bằng**, nhiều nút bên dưới được đếm lại ở từng tầng tổ tiên, nên tổng công việc
của kiểu baseline có dừng sớm này thường là `O(n log n)` và có thể đạt mức ấy.
Với **cây lệch hình dây**, gốc phát hiện chênh quá 1 và dừng ngay nên thực tế chỉ `O(n)` trong mã này.
**Lưu ý tinh tế:** nhiều tài liệu ghi `O(n²)` cho *phiên bản ngây thơ luôn kiểm tra mọi nút* trên cây dây;
không được lấy ví dụ ấy để khẳng định cận chặt `O(n²)` cho **mã đang dùng `return False` sớm**.

Ta muốn **mỗi cây con chỉ tính chiều cao một lần**, rồi trả kết quả cho cha.

In [12]:
def count_baseline_height_visits(root):
    calls = [0]
    def height(node):
        calls[0] += 1
        if node is None:
            return 0
        return 1 + max(height(node.left), height(node.right))
    def check(node):
        if node is None:
            return True
        if abs(height(node.left) - height(node.right)) > 1:
            return False
        return check(node.left) and check(node.right)
    answer = check(root)
    return answer, calls[0]

for n in [1,3,5,8]:
    root = None
    for v in range(n, 0, -1):
        root = TreeNode(v, root, None)
    print(f"Dây dài {n} nút -> answer, số lượt height:", count_baseline_height_visits(root))

Dây dài 1 nút -> answer, số lượt height: (True, 2)
Dây dài 3 nút -> answer, số lượt height: (False, 6)
Dây dài 5 nút -> answer, số lượt height: (False, 10)
Dây dài 8 nút -> answer, số lượt height: (False, 16)


### 4.2. Ý tưởng tối ưu: trả **hai thông tin** qua một đường đi

Đối với cha, ta cần biết:
1. Cây con **có cân bằng hay không**.
2. Nếu có cân bằng, **chiều cao** của cây con là bao nhiêu.

Có thể trả cặp `(height, is_balanced)`; hoặc dùng một giá trị đặc biệt `-1` làm “báo lỗi”.
**Vì sao `-1` an toàn?** Chiều cao hợp lệ không bao giờ âm (cây rỗng bằng 0),
nên `-1` chắc chắn không bị nhầm thành một độ cao bình thường.

Quy tắc: nhận `-1` từ bất kỳ con nào ⇒ trả `-1` cho cha ngay. Nếu hai con hợp lệ,
chênh >1 ⇒ trả `-1`. Nếu không, trả `1+max(L,R)`.
Đây là tư duy **postorder** (trái → phải → nút): dùng kết quả con trước khi quyết định tại cha.

In [13]:
height_left = 2
height_right = 1
print("Hai bên hợp lệ, chênh =", abs(height_left-height_right))
print("Chiều cao nút cha =", 1+max(height_left,height_right))
print("Nếu nhận tín hiệu con -1, cha phải trả -1 thay vì tiếp tục dùng max.")

Hai bên hợp lệ, chênh = 1
Chiều cao nút cha = 3
Nếu nhận tín hiệu con -1, cha phải trả -1 thay vì tiếp tục dùng max.


## 5. Vì sao cần chuyển sang kỹ thuật chính?

Ta chỉ cần tính chiều cao **một lần từ dưới lên**. Nếu cây con đã mất cân bằng, trả `-1` lập tức; cây cha không cần tính lại.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

DFS mỗi node trả `height` nếu cả cây con cân bằng; trả `-1` nếu mất cân bằng. Bước kết hợp kiểm tra: trái -1, phải -1, rồi `abs(left-right)>1`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

In [14]:
def solve(root: Optional[TreeNode]) -> bool:
    def height_or_bad(node):
        if node is None:return 0
        left=height_or_bad(node.left)
        if left==-1:return -1
        right=height_or_bad(node.right)
        if right==-1:return -1
        if abs(left-right)>1:return -1
        return 1+max(left,right)
    return height_or_bad(root)!=-1

### 6.1. Từng dòng của `height_or_bad(node)`

**Base case:** `node is None → 0`. Chỗ rỗng có độ cao 0 và là một cây cân bằng.

**Bước 1:** `left = height_or_bad(node.left)`.
Nếu `left == -1`, nhánh trái đã không cân ⇒ **return -1 ngay**, vì toàn cây cũng không thể cân.

**Bước 2:** gọi cho nhánh phải; nếu nhánh phải trả -1, truyền -1 về.

**Bước 3:** hai nhánh đều có chiều cao hợp lệ, tính `abs(left-right)`.
Nếu vượt 1 ⇒ nút hiện tại không cân bằng ⇒ `-1`.
Ngược lại trả chiều cao `1+max(left,right)`.

Cuối cùng `isBalanced(root)` trả `height_or_bad(root) != -1`.
Chú ý: hàm phụ *không trả bool*, mà trả **chiều cao hoặc mã lỗi**. Hàm ngoài mới trả `True/False` cho LeetCode.

In [15]:
print("Độ lệch 0/1 đều ổn:", abs(2-2)<=1, abs(2-3)<=1)
print("Độ lệch 2 không ổn:", abs(2-4)<=1)
print("-1 biểu diễn lỗi, không phải chiều cao:", -1 < 0)
assert solve(tree([3,9,20,None,None,15,7]))
assert not solve(bad_but_root_equal)

Độ lệch 0/1 đều ổn: True True
Độ lệch 2 không ổn: False
-1 biểu diễn lỗi, không phải chiều cao: True


## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Node | độ cao trái | độ cao phải | Kết quả |
|---|---:|---:|---:|
| lá `4` | 0 | 0 | 1 |
| `3` có một con sâu hơn | 1 | 0 | 2 |
| `2` với chênh lệch 2 | 2 | 0 | **-1** |
| cha của node lỗi | -1 | chưa cần xét | **-1** |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [16]:
def trace_height(node,depth=0):
    if node is None:return 0
    l=trace_height(node.left,depth+1)
    if l==-1:
        print('  '*depth,node.val,'nhận lỗi từ trái');return -1
    r=trace_height(node.right,depth+1)
    if r==-1:
        print('  '*depth,node.val,'nhận lỗi từ phải');return -1
    ans=-1 if abs(l-r)>1 else 1+max(l,r)
    print('  '*depth,f'node={node.val}, L={l}, R={r} -> {ans}')
    return ans
print('mã chiều cao cuối:',trace_height(tree([1,2,2,3,3,None,None,4,4])))

       node=4, L=0, R=0 -> 1
       node=4, L=0, R=0 -> 1
     node=3, L=1, R=1 -> 2
     node=3, L=0, R=0 -> 1
   node=2, L=2, R=1 -> 3
   node=2, L=0, R=0 -> 1
 node=1, L=3, R=1 -> -1
mã chiều cao cuối: -1


### 7.1. Bảng trace bottom-up: lỗi xuất hiện trước ở cây con

Xét `root=[1,2,2,3,3,None,None,4,4]`. Cây con trái của gốc (nút `2`) có:
- Một nhánh trái sâu 2 (`3→4`), nhánh phải sâu 1 (`3`) ⇒ **bản thân nút 2 vẫn cân bằng**.
- Nhưng tại **gốc 1**, cây con trái sâu 3 và cây con phải sâu 1 ⇒ lệch 2 ⇒ **không cân bằng**.

| Nút | Cao trái L | Cao phải R | Tính toán | Trả lên |
|---|---:|---:|---|---:|
| 4 (lá) | 0 | 0 | chênh 0 | 1 |
| 3 (bên trái) | 1 | 1 | chênh 0 | 2 |
| 3 (bên phải) | 0 | 0 | chênh 0 | 1 |
| 2 (con trái gốc) | 2 | 1 | chênh 1 | 3 |
| 2 (con phải gốc) | 0 | 0 | chênh 0 | 1 |
| **1 (gốc)** | **3** | **1** | **chênh 2 > 1** | **-1** |

Khi gặp `-1`, giá trị đó được **lan lên**, không được đem làm độ cao để lấy `max`.
Đọc trace in bên trên để tự kiểm thứ tự gọi và thời điểm trả.

### 7.2. Biến thể dễ hiểu hơn: trả về cặp `(height, balanced)`

Người mới có thể ngại `-1` vì nó là “mã đặc biệt” (sentinel). Ta có thể viết minh bạch hơn:
`visit(node)` trả `(height, ok)`, trong đó `ok` là boolean.

- Nút rỗng: `(0,True)`.
- Lấy cặp từ cả hai con: `(L,okL)` và `(R,okR)`.
- Tính `ok = okL and okR and abs(L-R)<=1`.
- Trả `(1+max(L,R), ok)`.

Cách này tính đầy đủ chiều cao ngay cả khi đã phát hiện không cân bằng; trong trường hợp xấu nhất vẫn `O(n)`.
Nó có thể dễ theo dõi bằng tay hơn, nhưng dùng một ít thông tin trong cặp tuple thay cho sentinel.

In [17]:
def is_balanced_pair(root):
    def visit(node):
        if node is None:
            return 0, True
        left_h, left_ok = visit(node.left)
        right_h, right_ok = visit(node.right)
        good_here = left_ok and right_ok and abs(left_h - right_h) <= 1
        return 1 + max(left_h, right_h), good_here
    return visit(root)[1]

for values in [[], [1], [3,9,20,None,None,15,7], [1,2,2,3,3,None,None,4,4]]:
    print(values, '->', is_balanced_pair(tree(values)))

[] -> True
[1] -> True
[3, 9, 20, None, None, 15, 7] -> True
[1, 2, 2, 3, 3, None, None, 4, 4] -> False


### 7.3. Trace kiểu cặp: thông tin nào đi từ cây con đến cha?

In kết quả **khi đệ quy quay trở lên**, không phải ngay khi mới vào nút.
Việc đợi cả hai cây con trả thông tin là bản chất của *postorder*: **trái, phải, nút**.
Hãy so với `trace_height` kiểu sentinel ở trên; hai cách phải đưa ra cùng kết luận.

In [18]:
def trace_balance_pair(root):
    def visit(node, indent=0):
        if node is None:
            return 0, True
        lh, lok = visit(node.left, indent+2)
        rh, rok = visit(node.right, indent+2)
        ok = lok and rok and abs(lh-rh) <= 1
        h = 1 + max(lh,rh)
        print(' '*indent + f"node={node.val}: L=({lh},{lok}), R=({rh},{rok}) => ({h},{ok})")
        return h, ok
    return visit(root)[1]

assert trace_balance_pair(tree([3,9,20,None,None,15,7]))
assert not trace_balance_pair(tree([1,2,2,3,3,None,None,4,4]))

  node=9: L=(0,True), R=(0,True) => (1,True)
    node=15: L=(0,True), R=(0,True) => (1,True)
    node=7: L=(0,True), R=(0,True) => (1,True)
  node=20: L=(1,True), R=(1,True) => (2,True)
node=3: L=(1,True), R=(2,True) => (3,True)
      node=4: L=(0,True), R=(0,True) => (1,True)
      node=4: L=(0,True), R=(0,True) => (1,True)
    node=3: L=(1,True), R=(1,True) => (2,True)
    node=3: L=(0,True), R=(0,True) => (1,True)
  node=2: L=(2,True), R=(1,True) => (3,True)
  node=2: L=(0,True), R=(0,True) => (1,True)
node=1: L=(3,True), R=(1,True) => (4,False)


## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Bất biến: tại mỗi node, kết quả là chiều cao **đúng** nếu toàn bộ cây con cân bằng, ngược lại là `-1`. Với cây rỗng đúng. Nếu hai cây con cân bằng và độ cao chênh ≤1, node hiện tại cũng cân bằng; còn lại chắc chắn sai. Quy nạp từ lá lên gốc.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [19]:
from typing import Optional
class Solution:
    def isBalanced(self, root: Optional[TreeNode]):
        def height_or_bad(node):
            if node is None:return 0
            left=height_or_bad(node.left)
            if left==-1:return -1
            right=height_or_bad(node.right)
            if right==-1:return -1
            if abs(left-right)>1:return -1
            return 1+max(left,right)
        return height_or_bad(root)!=-1

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [20]:
for a,expect in [([],True),([1],True),([3,9,20,None,None,15,7],True),([1,2,2,3,3,None,None,4,4],False),([1,2,None,3,None,4],False)]:
    assert solve(tree(a))==expect
    assert Solution().isBalanced(tree(a))==expect
print('PASS 5 ca cân bằng')

PASS 5 ca cân bằng


### 10.1. Ma trận test đối chiếu ba cách: baseline, sentinel và tuple

Trường hợp cần có:
- Cây rỗng và cây một nút (đương nhiên cân bằng).
- Cây có nhánh sâu 2 hơn nhánh còn lại (không cân).
- Cây **gốc có hai nhánh cao bằng nhau** nhưng **bên trong một cây con lệch**.
- Cây đủ ba tầng, cây không đầy đủ, nút âm/trùng số.

Ta dùng `baseline` làm cách kiểm tra đối chiếu (dù chậm hơn) trên bộ test nhỏ.
Code chính không được thay đổi dữ liệu trong cây khi kiểm tra.

In [21]:
balanced_test_cases = [
    ([], True), ([7], True), ([1,2,3], True),
    ([3,9,20,None,None,15,7], True),
    ([1,2,2,3,3,None,None,4,4], False),
    ([1,2,None,3,None,4], False),
    ([1,2,3,4,5,6,7], True),
    ([1,2,2,3,None,None,3,4], False),
]
for values, expected in balanced_test_cases:
    root = tree(values)
    assert baseline(root) == expected, (values, 'baseline')
    assert solve(root) == expected, (values, 'sentinel')
    assert is_balanced_pair(root) == expected, (values, 'pair')
    assert Solution().isBalanced(root) == expected, (values, 'Solution')
assert not baseline(bad_but_root_equal)
assert not solve(bad_but_root_equal)
assert not is_balanced_pair(bad_but_root_equal)
print(f"PASS {len(balanced_test_cases)+1} hình cây × 4 cách")

PASS 9 hình cây × 4 cách


### 10.2. Property: cân bằng không phụ thuộc `val`

Hai cây cùng các liên kết `.left/.right` nhưng gán các giá trị khác nhau phải có cùng trạng thái cân bằng.
Vì thuật toán không hề đọc `node.val`, mọi phép thay giá trị chỉ ảnh hưởng cách hiển thị, không ảnh hưởng kết luận.
Đây là một **property test** (kiểm thử tính chất) rất phù hợp với câu hỏi cấu trúc.

In [22]:
p = tree([3,9,20,None,None,15,7])
q = tree([-3,-9,-20,None,None,-15,-7])
assert show_tree(p) != show_tree(q)
assert solve(p) == solve(q) == True
bad_p = tree([1,2,None,3,None,4])
bad_q = tree([101,102,None,103,None,104])
assert solve(bad_p) == solve(bad_q) == False
print("PASS property: đổi toàn bộ val, kết luận không đổi")

PASS property: đổi toàn bộ val, kết luận không đổi


## 11. Phân tích Time/Space Complexity

`n` node. **Time O(n)** vì chiều cao của mỗi cây con được tính đúng một lần. **Space O(h)** recursion, tệ nhất O(n); baseline kiểm tra dừng sớm lặp lại việc tính chiều cao: **O(n log n)** trên cây cân bằng; với cây lệch, root có thể phát hiện sai ngay (**O(n)**). Bản ngây thơ cố kiểm tra tất cả nút trên cây lệch mới có cận **O(n²)**.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Chỉ so hai nhánh của node gốc.
- ❌ Không truyền tín hiệu -1 từ cây con lên cha.
- ❌ Thấy `left=-1` mà vẫn lấy `max(left,right)` như chiều cao hợp lệ.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Sai lầm kinh điển: kiểm tra mỗi gốc rồi kết luận

Đoạn **sai** chỉ tính chiều cao hai nhánh của gốc và bỏ qua độ cân bằng trong hai cây con:

`return abs(height(root.left)-height(root.right)) <= 1`

Cây `bad_but_root_equal` ở đầu bài có **hai nhánh gốc bằng nhau**, nên trả `True` theo code sai;
nhưng bên trong mỗi nhánh có nút lệch chiều cao >1. Muốn đúng, phải kiểm tra **tất cả cây con**.

Lỗi thứ hai: khi thấy `-1` trả lên từ trái thì vẫn tính `1+max(-1,right)`.
**Không bao giờ coi -1 là chiều cao**; -1 phải được truyền như thông báo lỗi và dừng sớm.

In [23]:
def wrong_root_only(root):
    def h(node):
        return 0 if node is None else 1 + max(h(node.left), h(node.right))
    if root is None:
        return True
    return abs(h(root.left) - h(root.right)) <= 1

print("Sai (chỉ xem gốc):", wrong_root_only(bad_but_root_equal))
print("Đúng (xem mọi nút):", solve(bad_but_root_equal))
assert wrong_root_only(bad_but_root_equal) and not solve(bad_but_root_equal)

Sai (chỉ xem gốc): True
Đúng (xem mọi nút): False


## 13. Nhận diện dạng bài và vận dụng

Bài cần vừa **tính số đo của cây** vừa **báo lỗi sớm**: trả sentinel từ dưới lên để không tính lại dữ liệu.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Dùng cây **một nhánh rất sâu** để chứng minh chỉ kiểm tra tại gốc chưa đủ; hãy lần theo điểm đầu tiên xuất hiện `-1`.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [24]:
root=tree([1,2,None,3,None,4]);print('cân bằng?',solve(root))

cân bằng? False


### 14.1. Khi gặp bài “truyền trạng thái lỗi” từ con lên cha

Một dạng tư duy hữu ích trong Tree DP (*Dynamic Programming trên cây*) là trả **thông tin đủ**
cho nút cha tính kết quả của nó. Ví dụ:

| Bài | Thông tin cây con trả về |
|---|---|
| #104 Maximum Depth | `height` (số nguyên không âm) |
| #110 Balanced Binary Tree | `height` hoặc `-1` báo lỗi |
| Biến thể đếm nút | `node_count` |
| Biến thể tổng các giá trị | `sum_of_values` |

Nhiều bài không cần tạo dict để ghi tất cả subtree; kết quả đệ quy trả về đã đủ.
Nếu đề yêu cầu “mọi nút phải hợp lệ” thì chỉ kiểm tra gốc thường **không đủ**.

**Độ phức tạp:** thuật toán sentinel đi qua mỗi nút tối đa một lần → `O(n)` time;
sử dụng call stack theo độ cao `h` → `O(h)` extra space (tệ nhất `O(n)` với cây lệch).

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Dự đoán `abs(2-3)`, `abs(1-4)` và `max(2,3)`; cho biết khi nào một node mất cân bằng.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [25]:
print(abs(2-3))
print(abs(1-4))
print(max(2,3))

1
3
3


### Bài 2 — Trace bằng tay (hiểu bản chất)

Tính chiều cao từ lá đi lên của cây `[1,2,2,3,3,None,None,4,4]`. Node nào quyết định cây không cân bằng?

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_is_balanced(root)` chạy O(n) bằng một lần duyệt. Hàm con `height` trả **-1** khi gặp cây con mất cân bằng, nếu không trả chiều cao.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [26]:
def exercise_is_balanced(root):
    # TODO: viết height(node), truyền -1 từ dưới lên nếu mất cân bằng.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Viết `exercise_height(root)` trả chiều cao bình thường, không dùng sentinel `-1`. So sánh khi nào dùng được lại kết quả chiều cao.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [27]:
def exercise_height(root):
    # TODO: root None -> 0; còn lại 1 + max(height con trái, con phải).
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_first_unbalanced(root)` trả **giá trị node đầu tiên mất cân bằng khi duyệt hậu tự trái → phải → gốc**; trả `None` nếu cây cân bằng. Trường hợp nhiều node cùng mất cân bằng, ưu tiên node thấp hơn ở nhánh trái.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [28]:
def exercise_first_unbalanced(root):
    # TODO: trả về cặp (height, bad_value) từ cây con; ưu tiên lỗi ở trái.
    raise NotImplementedError

### Bài 6 — Trả cả `height` và `is_balanced` trong một lần duyệt

Viết `exercise_balance_report(root)` trả `(height, is_balanced)` **kể cả khi cây không cân**.
Ví dụ cây `[1,2,None,3]` có height 3 và không cân bằng ⇒ `(3,False)`.

**Gợi ý 1:** giống biến thể tuple đã học, mọi cây con luôn trả một cặp kết quả.

<details><summary>Gợi ý 2 — base case</summary><code>None → (0, True)</code>.</details>

<details><summary>Gợi ý 3 — gộp</summary><code>(1+max(L,R), goodL and goodR and abs(L-R)<=1)</code>.</details>

In [29]:
def exercise_balance_report(root):
    # TODO: trả tuple (chiều cao, cây có cân bằng không?).
    # Không dùng hàm solve để tính lại cây nhiều lần.
    raise NotImplementedError

# Sau khi làm: assert exercise_balance_report(tree([1,2,None,3])) == (3, False)

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Cần kiểm tra chênh lệch tại **tất cả các node**, không chỉ gốc.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Từ lá đi lên tính `height`; dùng `-1` làm tín hiệu lỗi rồi trả ngược lên.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Nếu trái -1, trả -1; nếu phải -1 hoặc abs(trái-phải)>1, trả -1; ngược lại trả 1 + max.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Các giá trị: `1`, `3`, `3`. Một node mất cân bằng nếu hai cây con chênh lệch chiều cao **lớn hơn 1**.

### Bài 2 — Trace tay

Các lá `4` có cao 1; node `3` trái cao 2; node `2` trái cao 3; node `2` phải cao 1. Ở gốc `1`, hai phía cao `3` và `1`, chênh `2` nên **False**.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [30]:
def answer_is_balanced(root):
    def height(node):
        if node is None: return 0
        left=height(node.left)
        if left==-1: return -1
        right=height(node.right)
        if right==-1 or abs(left-right)>1: return -1
        return 1+max(left,right)
    return height(root)!=-1

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [31]:
def answer_height(root):
    if root is None: return 0
    return 1+max(answer_height(root.left),answer_height(root.right))

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [32]:
def answer_first_unbalanced(root):
    def visit(node):
        if node is None: return 0,None
        lh,bad_l=visit(node.left)
        if bad_l is not None: return 0,bad_l
        rh,bad_r=visit(node.right)
        if bad_r is not None: return 0,bad_r
        if abs(lh-rh)>1: return 0,node.val
        return 1+max(lh,rh),None
    return visit(root)[1]

### Đáp án bài 6 — trả tuple hai thông tin

Điểm khác sentinel: trả chiều cao bình thường ngay cả khi `balanced=False`.
Vì vậy hàm này cũng có thể dùng để **báo cáo** độ sâu của một cây không cân bằng.

In [33]:
def answer_balance_report(root):
    if root is None:
        return (0, True)
    lh, ok_l = answer_balance_report(root.left)
    rh, ok_r = answer_balance_report(root.right)
    height = 1 + max(lh, rh)
    balanced = ok_l and ok_r and abs(lh-rh) <= 1
    return height, balanced

assert answer_balance_report(None) == (0, True)
assert answer_balance_report(tree([1])) == (1, True)
assert answer_balance_report(tree([1,2,None,3])) == (3, False)
assert answer_balance_report(tree([3,9,20,None,None,15,7])) == (3, True)
print("PASS bài 6")

PASS bài 6


### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [34]:
for arr,expect in [([],True),([1],True),([3,9,20,None,None,15,7],True),([1,2,2,3,3,None,None,4,4],False)]:
    assert answer_is_balanced(tree(arr))==expect
assert answer_height(tree([1,2,None,3]))==3
assert answer_first_unbalanced(tree([1,2,2,3,3,None,None,4,4]))==1
assert answer_first_unbalanced(tree([1,2,3])) is None
print("PASS: đáp án 3 bài coding của phần luyện tập")

PASS: đáp án 3 bài coding của phần luyện tập


### Tự đánh giá #110 — phải nói rõ tại sao không dùng baseline để nộp

1. Vì sao kiểm tra mỗi gốc không đủ để kết luận toàn cây cân bằng?
2. Cây rỗng cao mấy? Cây chỉ có một nút có cân bằng không?
3. Vì sao có thể dùng `-1` làm dấu lỗi mà không nhầm với chiều cao thật?
4. Nếu nhánh trái trả `-1`, nên làm gì trước khi tính nhánh phải?
5. Baseline có kiểm tra dừng sớm thường đạt `O(n log n)` trên cây cân bằng; vì sao tối ưu được xuống `O(n)`?
6. Hàm trả `(height,boolean)` và hàm dùng sentinel khác nhau ở **kiểu giá trị trả** ra sao?

**Kiểm thử tự tạo:** một cây cân bằng đều 3 tầng, một cây hình dây 4 nút,
một cây gốc cân bằng nhưng nút sâu bị lệch. Chạy cả baseline và phương án tối ưu rồi lý giải từng kết quả.

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Ta chỉ cần tính chiều cao **một lần từ dưới lên**. Nếu cây con đã mất cân bằng, trả `-1` lập tức; cây cha không cần tính lại.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.